# Aula 01 - Lab 01: RAG com LangChain

**Curso:** Agentic Engineering - SkillGo
**Prof.:** Ives Santos

Neste laboratorio vamos construir, do zero, um pipeline de **RAG (Retrieval-Augmented Generation)**:

1. Configuracao do ambiente e da chave de API
2. Primeira chamada a um LLM com LangChain (`ChatAnthropic`)
3. Prompt Templates e LCEL (LangChain Expression Language)
4. Carregamento e *chunking* dos documentos
5. Embeddings (representacao vetorial do texto)
6. Criacao do banco vetorial (Chroma)
7. Retriever + Cadeia de RAG completa
8. Exercicios

> **Ideia central do RAG:** o LLM nao "sabe" sobre os seus dados privados. Em vez de re-treinar o modelo,
> nos *recuperamos* os trechos mais relevantes dos seus documentos e os injetamos no prompt como contexto.

---
## 1. Configuracao do ambiente

Instale as dependencias (rode uma unica vez). Se estiver no Colab, o `-q` deixa a saida limpa.

In [ ]:
# Descomente para instalar as dependencias
# %pip install -q langchain langchain-anthropic langchain-community langchain-chroma \
#     langchain-huggingface sentence-transformers chromadb python-dotenv

### 1.1 Chave de API

Crie um arquivo `.env` na raiz do projeto com:

```
ANTHROPIC_API_KEY=sua-chave-aqui
```

Nunca comite a chave no Git. Adicione `.env` ao `.gitignore`.

In [ ]:
import os
import getpass

from dotenv import load_dotenv

load_dotenv()  # carrega o .env, se existir

# Fallback: pede a chave de forma segura (nao aparece no notebook)
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

print("Chave configurada:", bool(os.environ.get("ANTHROPIC_API_KEY")))

---
## 2. Primeira chamada ao LLM

`ChatAnthropic` e o wrapper do LangChain para os modelos Claude.

| Parametro | O que faz |
|---|---|
| `model` | ID do modelo (ex.: `claude-opus-5`, `claude-sonnet-5`, `claude-haiku-4-5`) |
| `max_tokens` | Teto de tokens da resposta |
| `timeout` | Tempo maximo de espera |
| `max_retries` | Tentativas automaticas em erro transitorio |

In [ ]:
from langchain_anthropic import ChatAnthropic

MODEL_ID = "claude-opus-5"  # troque por "claude-sonnet-5" para reduzir custo

llm = ChatAnthropic(
    model=MODEL_ID,
    max_tokens=1024,
    timeout=60,
    max_retries=2,
)

resposta = llm.invoke("Explique em 2 frases o que e Agentic Engineering.")
print(resposta.content)

### 2.1 Anatomia da resposta

O retorno e um `AIMessage`. Alem do texto, ele traz metadados de uso (tokens) - util para
acompanhar custo.

In [ ]:
print("Tipo:", type(resposta).__name__)
print("Tokens de entrada :", resposta.usage_metadata["input_tokens"])
print("Tokens de saida   :", resposta.usage_metadata["output_tokens"])
print("Motivo da parada  :", resposta.response_metadata.get("stop_reason"))

### 2.2 Mensagens: System, Human, AI

O LangChain padroniza os papeis da conversa em classes de mensagem.
O `SystemMessage` define a *persona* e as regras do modelo.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

mensagens = [
    SystemMessage(content=(
        "Voce e um professor de engenharia de software. "
        "Responda de forma didatica, com no maximo 3 frases, sem jargao desnecessario."
    )),
    HumanMessage(content="Qual a diferenca entre um chatbot e um agente?"),
]

print(llm.invoke(mensagens).content)

### 2.3 Streaming

Em aplicacoes reais, o streaming melhora muito a percepcao de velocidade:
o usuario ve a resposta sendo escrita token a token.

In [ ]:
for pedaco in llm.stream("Liste 3 casos de uso de RAG em empresas."):
    print(pedaco.content, end="", flush=True)

---
## 3. Prompt Templates e LCEL

Um **PromptTemplate** e um prompt parametrizado. O **LCEL** (LangChain Expression Language)
permite compor componentes com o operador `|` (pipe), formando uma *chain*:

```
prompt | llm | parser
```

Cada elemento recebe a saida do anterior.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um especialista em {area}. Seja objetivo e pratico."),
    ("human", "{pergunta}"),
])

parser = StrOutputParser()  # extrai apenas o texto do AIMessage

chain = prompt | llm | parser

print(chain.invoke({
    "area": "arquitetura de sistemas de IA",
    "pergunta": "Quando NAO vale a pena usar RAG?",
}))

---
## 4. Documentos e Chunking

Nosso corpus de exemplo: um pequeno manual interno ficticio da empresa **SkillGo**.
Em um cenario real voce usaria *loaders* (`PyPDFLoader`, `WebBaseLoader`, `CSVLoader`, ...).

**Por que dividir em chunks?**
- O contexto do modelo e finito.
- Trechos menores geram embeddings mais precisos (menos "diluicao" semantica).
- Recuperamos so o que importa, reduzindo custo e ruido.

In [ ]:
from langchain_core.documents import Document

documentos_brutos = [
    Document(
        page_content=(
            "Politica de Ferias da SkillGo. Todo colaborador CLT tem direito a 30 dias de ferias "
            "por periodo aquisitivo de 12 meses. As ferias podem ser divididas em ate 3 periodos, "
            "sendo que um deles deve ter no minimo 14 dias corridos e os demais no minimo 5 dias. "
            "A solicitacao deve ser feita no portal RH com antecedencia minima de 30 dias. "
            "A venda de ate 10 dias (abono pecuniario) e permitida e deve ser sinalizada na solicitacao."
        ),
        metadata={"fonte": "manual_rh.pdf", "secao": "ferias"},
    ),
    Document(
        page_content=(
            "Politica de Home Office da SkillGo. O modelo padrao e hibrido, com 2 dias presenciais "
            "por semana definidos pelo time. Times de Engenharia podem optar por regime 100% remoto "
            "mediante aprovacao do gestor direto e do diretor da area. O auxilio home office e de "
            "R$ 250 mensais e cobre internet e energia. Equipamentos sao fornecidos pela empresa e "
            "devem ser devolvidos no desligamento."
        ),
        metadata={"fonte": "manual_rh.pdf", "secao": "home_office"},
    ),
    Document(
        page_content=(
            "Processo de Deploy da SkillGo. Todo deploy em producao exige pull request aprovado por "
            "no minimo 2 revisores, pipeline de CI verde e cobertura de testes acima de 80%. "
            "A janela de deploy vai de segunda a quinta, das 9h as 17h. Sextas-feiras e vesperas de "
            "feriado sao bloqueadas. Deploys emergenciais exigem aprovacao do tech lead de plantao "
            "e abertura de incidente retroativo em ate 24 horas."
        ),
        metadata={"fonte": "runbook_eng.md", "secao": "deploy"},
    ),
    Document(
        page_content=(
            "Politica de uso de IA na SkillGo. E permitido usar assistentes de IA para gerar codigo, "
            "desde que o resultado passe por revisao humana. E proibido enviar dados de clientes, "
            "credenciais, chaves de API ou codigo de modulos classificados como confidenciais para "
            "ferramentas de IA externas. O uso de modelos hospedados na infraestrutura da empresa "
            "nao possui essa restricao."
        ),
        metadata={"fonte": "politica_ia.md", "secao": "ia"},
    ),
    Document(
        page_content=(
            "Onboarding de Engenharia da SkillGo. O novo colaborador recebe notebook e acessos no "
            "primeiro dia. Na primeira semana deve concluir o treinamento de seguranca, configurar o "
            "ambiente local e abrir seu primeiro pull request, ainda que pequeno. Cada pessoa recebe "
            "um buddy que acompanha os primeiros 30 dias. A avaliacao de experiencia ocorre em 45 e 90 dias."
        ),
        metadata={"fonte": "onboarding_eng.md", "secao": "onboarding"},
    ),
]

print(f"{len(documentos_brutos)} documentos carregados.")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,       # tamanho alvo do chunk, em caracteres
    chunk_overlap=60,     # sobreposicao: evita cortar uma ideia ao meio
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documentos_brutos)

print(f"{len(documentos_brutos)} documentos -> {len(chunks)} chunks\n")
print("--- Exemplo de chunk ---")
print(chunks[0].page_content)
print("\nMetadados:", chunks[0].metadata)

---
## 5. Embeddings

Um **embedding** e um vetor de numeros que representa o *significado* de um texto.
Textos com sentidos parecidos ficam proximos nesse espaco vetorial.

A Anthropic nao oferece API de embeddings, entao usamos um modelo aberto rodando
**localmente** via `sentence-transformers`. Vantagens didaticas: gratuito, offline e sem
chave adicional.

> O modelo `paraphrase-multilingual-MiniLM-L12-v2` funciona bem em portugues.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},
)

vetor = embeddings.embed_query("Quantos dias de ferias eu tenho?")
print("Dimensao do vetor:", len(vetor))
print("Primeiros 8 valores:", [round(v, 4) for v in vetor[:8]])

### 5.1 Intuicao: similaridade entre frases

Vamos comparar frases usando **similaridade de cosseno**. Como normalizamos os vetores,
o cosseno e simplesmente o produto escalar.

In [ ]:
import numpy as np

def similaridade(a: str, b: str) -> float:
    va, vb = embeddings.embed_query(a), embeddings.embed_query(b)
    return float(np.dot(va, vb))

pares = [
    ("Quantos dias de ferias tenho?", "Politica de ferias e periodo aquisitivo"),
    ("Quantos dias de ferias tenho?", "Como faco deploy em producao?"),
    ("Posso trabalhar de casa?", "Regime hibrido e auxilio home office"),
]

for a, b in pares:
    print(f"{similaridade(a, b):.3f}  |  '{a}'  <->  '{b}'")

---
## 6. Banco Vetorial (Vector Store)

O banco vetorial armazena os embeddings dos chunks e faz **busca por similaridade**
de forma eficiente. Aqui usamos o **Chroma** em memoria.

Alternativas comuns: FAISS (local), Pinecone, Qdrant, Weaviate, pgvector.

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="skillgo_manual",
)

print("Documentos indexados:", vectorstore._collection.count())

In [ ]:
# Busca direta no banco vetorial, com o score de distancia
pergunta = "Posso vender parte das minhas ferias?"

for doc, score in vectorstore.similarity_search_with_score(pergunta, k=3):
    print(f"[score={score:.4f}] ({doc.metadata['secao']}) {doc.page_content[:110]}...\n")

---
## 7. Retriever

O **retriever** e a interface padrao de recuperacao do LangChain. Ele encapsula o
vector store e pode ser plugado diretamente em uma chain.

Estrategias de busca:
- `similarity` - os k mais parecidos (padrao)
- `mmr` - *Maximal Marginal Relevance*: equilibra relevancia e diversidade
- `similarity_score_threshold` - descarta resultados abaixo de um limiar

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

encontrados = retriever.invoke("Qual o processo para subir codigo em producao?")

for i, doc in enumerate(encontrados, 1):
    print(f"{i}. ({doc.metadata['fonte']} / {doc.metadata['secao']}) {doc.page_content[:100]}...")

---
## 8. A cadeia de RAG completa

Agora juntamos tudo. O fluxo:

```
pergunta -> retriever -> contexto
                             \
                              +-> prompt -> LLM -> resposta
                             /
pergunta --------------------
```

O `RunnableParallel` (aqui na forma de um dicionario) executa a recuperacao e o
repasse da pergunta em paralelo, entregando ambos ao prompt.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

TEMPLATE_RAG = """Voce e o assistente interno da SkillGo.
Responda a pergunta do colaborador usando EXCLUSIVAMENTE o contexto abaixo.

Regras:
- Se a resposta nao estiver no contexto, diga: "Nao encontrei essa informacao nos documentos internos."
- Nunca invente politicas, prazos ou valores.
- Cite a secao de origem entre colchetes ao final, ex.: [ferias].
- Seja direto: no maximo 4 frases.

<contexto>
{contexto}
</contexto>

Pergunta: {pergunta}
"""

prompt_rag = ChatPromptTemplate.from_template(TEMPLATE_RAG)


def formatar_docs(docs) -> str:
    """Concatena os chunks recuperados em um unico bloco de texto."""
    return "\n\n".join(
        f"[{d.metadata.get('secao', 'geral')}] {d.page_content}" for d in docs
    )


rag_chain = (
    {"contexto": retriever | formatar_docs, "pergunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | parser
)

print(rag_chain.invoke("Posso dividir minhas ferias em quantos periodos?"))

### 8.1 Bateria de testes

Repare no ultimo caso: a pergunta esta fora do corpus. Um RAG bem construido
**admite que nao sabe** em vez de alucinar.

In [ ]:
perguntas = [
    "Qual o valor do auxilio home office?",
    "Posso fazer deploy na sexta-feira?",
    "Posso colar codigo de cliente no ChatGPT?",
    "Quando acontece a avaliacao de experiencia?",
    "Qual e o plano de saude da empresa?",  # <- nao existe no corpus
]

for p in perguntas:
    print(f"P: {p}")
    print(f"R: {rag_chain.invoke(p)}\n" + "-" * 70)

### 8.2 RAG com fontes

Em producao, quase sempre queremos devolver **a resposta e as fontes** usadas.
Para isso, mantemos os documentos recuperados vivos ate o final da cadeia.

In [ ]:
from langchain_core.runnables import RunnableParallel

rag_com_fontes = RunnableParallel(
    {"docs": retriever, "pergunta": RunnablePassthrough()}
).assign(
    resposta=(
        lambda x: {"contexto": formatar_docs(x["docs"]), "pergunta": x["pergunta"]}
    )
    | prompt_rag
    | llm
    | parser
)

saida = rag_com_fontes.invoke("Quantos revisores um pull request precisa?")

print("RESPOSTA:", saida["resposta"], "\n")
print("FONTES:")
for d in saida["docs"]:
    print(f"  - {d.metadata['fonte']} (secao: {d.metadata['secao']})")

---
## 9. Exercicios

1. **Chunking.** Troque `chunk_size` para `800` e depois para `150`. Reindexe e refaca as
   perguntas. O que muda na qualidade das respostas? Por que?
2. **k do retriever.** Teste `k=1` e `k=5`. Qual o efeito no custo (tokens de entrada) e na
   precisao?
3. **MMR.** Troque `search_type="similarity"` por `"mmr"` com
   `search_kwargs={"k": 3, "fetch_k": 10}`. Compare os documentos recuperados.
4. **Novo documento.** Acrescente um documento sobre plano de saude ao corpus, reindexe
   e verifique se a ultima pergunta da bateria passa a ser respondida.
5. **Prompt.** Modifique o `TEMPLATE_RAG` para que a resposta sempre inclua um nivel de
   confianca (alto/medio/baixo) baseado na aderencia do contexto a pergunta.
6. **Loader real.** Substitua os documentos manuais por um PDF usando `PyPDFLoader`
   (`pip install pypdf`).

---
## Resumo

| Etapa | Componente LangChain |
|---|---|
| Modelo de linguagem | `ChatAnthropic` |
| Prompt parametrizado | `ChatPromptTemplate` |
| Composicao | LCEL (`\|`), `RunnablePassthrough`, `RunnableParallel` |
| Divisao de texto | `RecursiveCharacterTextSplitter` |
| Vetorizacao | `HuggingFaceEmbeddings` |
| Banco vetorial | `Chroma` |
| Recuperacao | `vectorstore.as_retriever()` |
| Saida em texto | `StrOutputParser` |

**Proximo laboratorio:** `Aula01_Lab02_Tools` - dando ferramentas ao modelo para que ele
deixe de apenas responder e passe a **agir**.